In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

flight_radar_api_key = os.getenv("FLIGHT_RADAR_API_KEY")


In [ ]:
from datetime import datetime, timedelta


def generate_timestamps(datetime_1: datetime, datetime_2: datetime, interval_minutes: int):
    """
    Generate Unix timestamps between two datetime objects spaced every `interval_minutes`.

    Args:
        datetime_1 (datetime): Start datetime.
        datetime_2 (datetime): End datetime.
        interval_minutes (int): Interval in minutes.

    Returns:
        List[int]: List of Unix timestamps (seconds since epoch).
    """
    timestamps = []
    current = datetime_1
    while current <= datetime_2:
        timestamps.append(int(current.timestamp()))
        current += timedelta(minutes=interval_minutes)
    return timestamps


start = datetime(2025, 4, 23, 7, 10)  # April 23, 2025 00:00
end = datetime(2025, 4, 23, 12, 0)    # April 23, 2025 12:00

timestamps = generate_timestamps(start, end, interval_minutes=10)
print(timestamps)

In [ ]:
import requests
import time

all_data = {}

def get_position(timestamp: str):
  url = "https://fr24api.flightradar24.com/api/historic/flight-positions/light"
  params = {
    'flights': 'LY385',
    'timestamp': timestamp
  }
  headers = {
    'Accept': 'application/json',
    'Accept-Version': 'v1',
    'Authorization': f'Bearer {flight_radar_api_key}'
  }

  try:
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()
    return data
  except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
  except Exception as err:
      print(f"An error occurred: {err}")

for timestamp in timestamps:
  data = get_position(timestamp)
  print(data)
  # time.sleep(2.5)
  # if not data:
  #   continue
  # else:
  #   all_data[str(datetime.fromtimestamp(timestamp))] = get_position(timestamp)


In [ ]:
data

In [ ]:
import requests
import os
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()

API_TOKEN = os.getenv("FLIGHT_RADAR_API_KEY_2")

BASE_URL = "https://fr24api.flightradar24.com/api/flight-summary/light"

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Accept": "application/json",
    "Accept-Version": "v1"
}


# ---------------------------------------------------
# 1️⃣ Fetch all flights (14-day chunks)
# ---------------------------------------------------
def fetch_all_flights(origin, destination, start_date, end_date):

    all_flights = []
    current_start = start_date

    while current_start < end_date:

        current_end = min(current_start + timedelta(days=14), end_date)

        params = {
            "routes": f"{origin}-{destination}",
            "flight_datetime_from": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "flight_datetime_to": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "limit": 20000,
            "sort": "asc"
        }

        print(f"Fetching {params['flight_datetime_from']} → {params['flight_datetime_to']}")

        response = requests.get(BASE_URL, headers=HEADERS, params=params)

        if response.status_code != 200:
            print("Error:", response.status_code, response.text)
            break

        data = response.json().get("data", [])
        all_flights.extend(data)

        current_start = current_end
        time.sleep(0.3)  # avoid rate limit

    return all_flights


# ---------------------------------------------------
# 2️⃣ Extract airborne intervals
# ---------------------------------------------------
def extract_intervals(flights):

    intervals = []

    for f in flights:
        takeoff = f.get("first_seen")
        landed = f.get("last_seen")

        if takeoff and landed:
            start = datetime.strptime(takeoff, "%Y-%m-%dT%H:%M:%SZ")
            end = datetime.strptime(landed, "%Y-%m-%dT%H:%M:%SZ")
            intervals.append((start, end))

    return intervals


# ---------------------------------------------------
# 3️⃣ Generate global 4-minute timeline
# ---------------------------------------------------
def generate_global_timeline(start_date, end_date):

    timestamps = []
    current = start_date

    while current <= end_date:
        timestamps.append(current)
        current += timedelta(minutes=4)

    return timestamps


# ---------------------------------------------------
# 4️⃣ Keep only timestamps with active flight
# ---------------------------------------------------
def filter_active_timestamps(timeline, intervals):

    active = []

    for t in timeline:
        for start, end in intervals:
            if start <= t <= end:
                active.append(t)
                break

    return active


# ---------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------

START = datetime(2025, 4, 1, 0, 0)
END = datetime(2025, 6, 19, 23, 59)

# 1️⃣ Get all BCN→MUC flights
flights = fetch_all_flights("BCN", "MUC", START, END)

print("\nTotal flights collected:", len(flights))

# 2️⃣ Extract airborne intervals
intervals = extract_intervals(flights)

print("Total valid airborne intervals:", len(intervals))

# 3️⃣ Generate full 4-min grid
timeline = generate_global_timeline(START, END)

print("Total 4-min timestamps in period:", len(timeline))

# 4️⃣ Filter only timestamps where at least one flight is airborne
active_timestamps = filter_active_timestamps(timeline, intervals)

print("Active timestamps (with flight flying):", len(active_timestamps))

# Convert to string format if needed
active_timestamps_str = [
    t.strftime("%Y-%m-%dT%H:%M:%SZ") for t in active_timestamps
]

print("\nExample timestamps:")
print(active_timestamps_str[:10])


In [ ]:
# Convert datetime objects to Unix timestamps (UTC)
active_unix_timestamps = [
    int(t.timestamp()) for t in active_timestamps
]

print(active_unix_timestamps[:10])
len(active_unix_timestamps)

In [ ]:
active_timestamps_str

In [ ]:
flights

In [ ]:
import requests
import time
from datetime import datetime, timedelta

import os
from dotenv import load_dotenv

load_dotenv()

flight_radar_api_key = os.getenv("FLIGHT_RADAR_API_KEY")


def generate_timestamps(start_date: datetime, end_date: datetime, interval_seconds: int) -> list[int]:
    timestamps = []
    current_time = start_date
    delta = timedelta(seconds=interval_seconds)

    while current_time <= end_date:
        timestamps.append(int(current_time.timestamp()))
        current_time += delta
    return timestamps


def fetch_historic_flight_positions(api_token: str, timestamps: list[int], **filters) -> dict[str, list]:
    """
    Fetches historical flight positions over a date range.

    Parameters:
        api_token (str): Your Flightradar24 API token.
        start_date (datetime): The start date and time.
        end_date (datetime): The end date and time.
        interval_seconds (int): Time interval between data points in seconds. Default is 15 minutes (900 seconds).
        **filters: Endpoint filters like bounds, flights, callsigns, registrations, limit etc.

    Returns:
        List[dict]: A list of flight position data.
    """
    api_url = 'https://fr24api.flightradar24.com/api/historic/flight-positions/full'
    headers = {
        'Accept': 'application/json',
        'Accept-Version': 'v1',
        'Authorization': f'Bearer {flight_radar_api_key}'
    }

    all_data = {}

    for ts in timestamps:
        dt_utc = datetime.fromtimestamp(ts)
        print(dt_utc)
        params = {'timestamp': ts}
        params.update(filters)  # Add additional filters if provided

        response = requests.get(api_url, headers=headers, params=params)
        if response.status_code == 429:
            print(f"Rate limit reached. Sleeping for {response.headers.get('Retry-After', 60)} seconds.")
            time.sleep(int(response.headers.get('Retry-After', 60)))
            response = requests.get(api_url, headers=headers, params=params)
                

        data = response.json()
        if not data:
            print("no data")
            continue
        data = data.get('data')
        if not data:
            continue
        print("DATA EXISTS !!")
        for element in data:
            flight_number = element.get('flight')
            if not flight_number:
                continue
            if flight_number in all_data:
                all_data[flight_number].append(element)
            else:
                all_data[flight_number] = [element]

    return all_data

        

In [ ]:

API_TOKEN = flight_radar_api_key
routes = 'BCN-MUC'
interval_seconds = 4 * 60  # 4 minutes

# Define date range
start_date = datetime(2025, 4, 1, 0, 0)
end_date = datetime(2025, 6, 19, 23, 59)

timestamps = generate_timestamps(start_date, end_date, interval_seconds)

# Fetch data
flight_data = fetch_historic_flight_positions(
    api_token=API_TOKEN,
    timestamps=timestamps,
    routes=routes,
    limit=1000
)
print(flight_data)

In [ ]:
import json

# Write to a file
with open('data/BCN-MUC/flight_data_BCN-MUC.json', 'w') as f:
    json.dump(flight_data, f, indent=4)

In [ ]:
import requests
import time
from datetime import datetime, timedelta

def generate_timestamps(start_date: datetime, end_date: datetime, interval_seconds: int) -> list[int]:
    timestamps = []
    current_time = start_date
    delta = timedelta(seconds=interval_seconds)

    while current_time <= end_date:
        timestamps.append(int(current_time.timestamp()))
        current_time += delta
    return timestamps


def fetch_historic_flight_positions(api_token: str, timestamps: list[int], **filters) -> dict[str, list]:
    """
    Fetches historical flight positions over a date range.

    Parameters:
        api_token (str): Your Flightradar24 API token.
        start_date (datetime): The start date and time.
        end_date (datetime): The end date and time.
        interval_seconds (int): Time interval between data points in seconds. Default is 15 minutes (900 seconds).
        **filters: Endpoint filters like bounds, flights, callsigns, registrations, limit etc.

    Returns:
        List[dict]: A list of flight position data.
    """
    api_url = 'https://fr24api.flightradar24.com/api/historic/flight-positions/full'
    headers = {
        'Accept': 'application/json',
        'Accept-Version': 'v1',
        'Authorization': f'Bearer {flight_radar_api_key}'
    }

    all_data = {}

    for ts in timestamps:
        dt_utc = datetime.fromtimestamp(ts)
        print(dt_utc)
        params = {'timestamp': ts}
        params.update(filters)  # Add additional filters if provided

        response = requests.get(api_url, headers=headers, params=params)
        if response.status_code != 200:
            if response.status_code == 429:
                print(f"Rate limit reached. Sleeping for {response.headers.get('Retry-After', 60)} seconds.")
                time.sleep(int(response.headers.get('Retry-After', 60)))
                response = requests.get(api_url, headers=headers, params=params)
            else:
                print("Error Status:sleeping for 10 seconds.")
                time.sleep(10)
                response = requests.get(api_url, headers=headers, params=params)

        data = response.json()
        if not data:
            continue
        data = data.get('data')
        if not data:
            continue

        for element in data:
            flight_number = element.get('flight')
            if not flight_number:
                continue
            if flight_number in all_data:
                all_data[flight_number].append(element)
            else:
                all_data[flight_number] = [element]

    return all_data

# Example usage
if __name__ == '__main__':
    # Your API token
    API_TOKEN = flight_radar_api_key
    routes = 'CDG-JFK'
    interval_seconds = 4 * 60  # 15 minutes

    # Define date range
    start_date = datetime(2025, 3, 1, 0, 0)  # April 23, 2025 00:00
    end_date = datetime(2025, 6, 6, 23, 59)

    timestamps = generate_timestamps(start_date, end_date, interval_seconds)

    # Fetch data
    flight_data = fetch_historic_flight_positions(
        api_token=API_TOKEN,
        timestamps=timestamps,
        routes=routes,
        limit=1000
    )
    print(flight_data)

    import json

    # Write to a file
    with open('flight_data_CDG_JFK.json', 'w') as f:
        json.dump(flight_data, f, indent=4)

        

In [ ]:
flight_data


In [ ]:
import json

# Write to a file
with open('flight_data.json', 'w') as f:
    json.dump(flight_data, f, indent=4)

NOW WE WILL SEPARATE THE FLIGHTS

In [ ]:
import json

with open('flight_data_CDG_FCO.json', 'r') as f:
    flight_datas = json.load(f)

In [ ]:
list_flight = flight_datas["AZ317"]

max_date = max([element["timestamp"] for element in list_flight])

max_date

Lets organize the flights based on the flights unique ids

In [ ]:
import json

with open('flight_data_CDG_FCO.json', 'r') as f:
    flight_data_1 = json.load(f)

with open('flight_data_CDG_FCO_2.json', 'r') as f:
    flight_data_2 = json.load(f)

organized_flight_data = {}
fr_24_ids = []

def separate_distinct_flights(flight_data, fr_24_ids, organized_flight_data):
    for flight_number, elements in flight_data.items():
        if flight_number not in organized_flight_data:
            organized_flight_data[flight_number] = {}
        for coordinate in elements:
            if coordinate["fr24_id"] in fr_24_ids:
                last_timestamp = organized_flight_data[flight_number][coordinate["fr24_id"]][-1]["timestamp"]
                if last_timestamp != coordinate["timestamp"]:
                    organized_flight_data[flight_number][coordinate["fr24_id"]].append(coordinate)
            else:
                organized_flight_data[flight_number][coordinate["fr24_id"]] = [coordinate]
                fr_24_ids.append(coordinate["fr24_id"])

In [ ]:
organized_flight_data

In [ ]:
separate_distinct_flights(flight_data_1, fr_24_ids, organized_flight_data)
separate_distinct_flights(flight_data_2, fr_24_ids, organized_flight_data)

In [ ]:
from copy import deepcopy

STATIC_KEYS = [
    'flight', 'callsign', 'source', 'hex', 'type', 'reg',
    'painted_as', 'operating_as', 'orig_iata', 'orig_icao',
    'dest_iata', 'dest_icao', 'eta'
]

def transform_flight_data_to_list(data):
    result = []

    for flight_ids in data.values():  # skip flight_number key
        for flight_id, records in flight_ids.items():
            if not records:
                continue  # skip empty lists

            # Use first record to extract metadata
            metadata = {key: records[0].get(key) for key in STATIC_KEYS}
            metadata['fr24_id'] = flight_id  # include flight_id in metadata

            # Build dynamic flight data without static keys
            dynamic_data = [
                {k: v for k, v in record.items() if k not in STATIC_KEYS}
                for record in records
            ]

            result.append({
                "flight_metadata": metadata,
                "flight_data": dynamic_data
            })

    return result


In [ ]:
transformed_list = transform_flight_data_to_list(organized_flight_data)

transformed_list

In [ ]:
len(transformed_list)

In [ ]:
import json

with open('flight_data.json', 'w') as f:
    json.dump(transformed_list, f, indent=4)

In [ ]:
import pandas as pd
import datetime
from math import radians, sin, cos, sqrt, atan2

def round_down_to_even_hour(dt):
    """Round down to the most recent even hour (causal logic)."""
    hour = dt.hour
    even_hour = hour - (hour % 2)  # floor to nearest even hour
    return dt.replace(hour=even_hour, minute=0, second=0, microsecond=0)


def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in km between two points."""
    R = 6371  # Earth radius in kilometers
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlambda / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def enrich_flight_data_with_weather(flight_entries, meteo_df):
    """Attach weather info to each point in flight_data list."""
    # Convert validdate to datetime and set as index
    meteo_df['validdate'] = pd.to_datetime(meteo_df['validdate'])
    meteo_df = meteo_df.set_index('validdate')

    total = 0
    for entry in flight_entries:
        for point in entry["flight_data"]:
            total += 1
    print(total)
    counter = 0
    for entry in flight_entries:
        for point in entry["flight_data"]:
            counter += 1
            print(f"{counter}/{total}")
            try:
                # Parse timestamp
                if point["timestamp"] < '2025-04-01':
                    continue
                ts = pd.to_datetime(point["timestamp"])
                rounded_ts = round_down_to_even_hour(ts)
                
                if rounded_ts not in meteo_df.index:
                    continue  # skip if no weather at this time

                # Subset with matching datetime
                subset = meteo_df.loc[[rounded_ts]]

                # Find closest point by haversine distance
                lat, lon = point['lat'], point['lon']
                subset = subset.copy()
                subset["distance"] = subset.apply(
                    lambda row: haversine(lat, lon, row["lat"], row["lon"]), axis=1
                )
                closest_row = subset.sort_values("distance").iloc[0]

                # Attach weather data
                weather_data = closest_row.drop("distance").to_dict()
                point["weather"] = weather_data

            except Exception as e:
                print(f"Error processing point {point}: {e}")
                point["weather"] = None  # in case of error, keep it explicit

    return flight_entries


In [ ]:
import json
# Load meteorological data
meteo_df = pd.read_csv("merged_meteomatics_data.csv")
with open('flight_data.json', 'r') as f:
    transformed_list = json.load(f)

In [ ]:
import pandas as pd

def filter_flights_after_cutoff(flight_entries, cutoff="2025-04-01T00:00:00Z"):
    cutoff_dt = pd.to_datetime(cutoff)
    filtered = []
    counter = 1
    total = len(flight_entries)
    for entry in flight_entries:
        print(f"{counter}/{total}")
        counter += 1
        timestamps = [pd.to_datetime(p['timestamp']) for p in entry["flight_data"]]
        if all(ts >= cutoff_dt for ts in timestamps):
            filtered.append(entry)
    return filtered

In [ ]:
filtered_flight_entries = filter_flights_after_cutoff(transformed_list)

In [ ]:
len(filtered_flight_entries)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from scipy.spatial import cKDTree

def round_down_to_even_hour(dt):
    hour = dt.hour - (dt.hour % 2)
    return dt.replace(hour=hour, minute=0, second=0, microsecond=0)

def preprocess_meteo_data(meteo_df):
    meteo_df['validdate'] = pd.to_datetime(meteo_df['validdate'])
    grouped = {}
    for ts, group in meteo_df.groupby('validdate'):
        coords = group[['lat', 'lon']].to_numpy()
        tree = cKDTree(coords)
        grouped[ts] = (group.reset_index(drop=True), tree)
    return grouped

def enrich_flight_data_with_weather(flight_entries, meteo_by_ts):
    enriched = []
    total = 0
    for entry in flight_entries:
        for point in entry["flight_data"]:
            total += 1
    print(total)
    counter = 0
    for entry in flight_entries:
        for point in entry["flight_data"]:
            counter += 1
            print(f"{counter}/{total}")
            ts = pd.to_datetime(point["timestamp"])
            rounded_ts = round_down_to_even_hour(ts)

            if rounded_ts not in meteo_by_ts:
                continue

            df, tree = meteo_by_ts[rounded_ts]
            dist, idx = tree.query([point["lat"], point["lon"]], k=1)
            weather_row = df.iloc[idx].to_dict()
            point["weather"] = weather_row

        enriched.append(entry)

    return enriched


In [ ]:
# Assuming transformed_list is your flight data from earlier
meteo_by_ts = preprocess_meteo_data(meteo_df)

# Load and enrich
# with open("transformed_flight_data.json") as f:
#     flight_entries = json.load(f)
enriched = enrich_flight_data_with_weather(filtered_flight_entries, meteo_by_ts)


# Save it if needed
# import json
# with open("enriched_flight_data.json", "w") as f:
#     json.dump(enriched_list, f, indent=4)


In [ ]:
len(enriched_list)

In [ ]:
import json

with open("enriched_flight_data.json") as f:
    flight_entries = json.load(f)

In [ ]:
def clean_weather_fields(flight_entries):
    for entry in flight_entries:
        for point in entry.get("flight_data", []):
            point.pop('fr24_id', None)
            weather = point.get("weather")
            if weather:
                for key in ['validdate', 'lat', 'lon']:
                    weather.pop(key, None)

clean_weather_fields(flight_entries)

In [ ]:
enriched_list[0]

In [ ]:
import json
with open("enriched_flight_data.json", "w") as f:
    json.dump(enriched_list, f, indent=4)

In [ ]:
import json
from datetime import datetime
import pandas as pd

def compress_flight_data(flight_entries):
    compressed = []

    for entry in flight_entries:
        meta = entry["flight_metadata"]
        compact_data = []
        for point in entry["flight_data"]:
            w = point.get("weather", {})
            row = [point.get("lat"), point.get("lon"), point.get("track"), point.get("alt"), point.get("gspeed"), point.get("vspeed"), point.get('timestamp'), w.get("edr_300hPa:m23s1"), w.get("turbulence_ellrod_300hPa:s2"), w.get("richardson_number_300hPa:idx"), w.get("turbulence_cape:m23s"), w.get("wind_speed_10000m:ms"), w.get("wind_dir_10000m:d"), w.get("wind_speed_2m:ms"), w.get("wind_dir_2m:d")]
            compact_data.append(row)

        compressed.append({
            "flight_metadata": meta,
            "flight_data": compact_data
        })

    return compressed


In [ ]:
light_data = compress_flight_data(flight_entries)

print(light_data)

In [ ]:
import json
with open("light_flight_data.json", "w") as f:
    json.dump(light_data, f, indent=2)

I NOTICED THAT SOME OF THE FLIGHTS DATAPOINTS WERE NOT ORDERED WELL BASED ON THEIR TIMESTAMP

In [ ]:
import json

def remove_unsorted_flights(input_path, output_path):
    with open(input_path, "r") as f:
        data = json.load(f)

    cleaned_data = []
    removed_flights = []

    for flight in data:
        flight_id = flight["flight_metadata"].get("fr24_id", "Unknown")
        flight_number = flight["flight_metadata"].get("flight", "Unknown")
        timestamps = [row[6] for row in flight["flight_data"]]

        if timestamps != sorted(timestamps):
            print(f"❌ Removing flight {flight_number} (ID: {flight_id}) due to unordered timestamps.")
            removed_flights.append(flight_id)
        else:
            cleaned_data.append(flight)

    with open(output_path, "w") as f:
        json.dump(cleaned_data, f, indent=2)

    print(f"\n✅ Saved {len(cleaned_data)} sorted flights.")
    print(f"🗑️ Removed {len(removed_flights)} unsorted flights.")

# Example usage
remove_unsorted_flights(
    input_path="light_flight_data.json",
    output_path="final_data.json"
)

In [ ]:
import json

with open("final_data.json", "r") as f:
    flights_data = json.load(f)

print(flights_data[0])

In [ ]:
!pip install openai==0.28

NOW LETS BUILD THE TEXT REPRESENTATION OF THE FLIGHT

In [ ]:
import openai
import os

# Set your OpenAI API key
openai.api_key = os.getenv("OPENAI_API_KEY")
# Define the query
response = openai.ChatCompletion.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What's the capital of Japan?"}
    ],
    temperature=0.7
)

# Print the response
print(response['choices'][0]['message']['content'])


NOW WE WILL START CALLING OPENAI IN A CHAIN USING LANGCHAIN, TO MAKE CALLS IN BATCHES

In [ ]:
!pip install langchain==0.0.340

In [ ]:
flights_data_sample = flights_data[:1]

In [ ]:
import os
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o-2024-11-20", openai_api_key=os.getenv("OPENAI_API_KEY_2"), temperature=0.5, max_tokens=5000)

prompt="""You are given a flight record consisting of:

1. `flight_metadata`: A dictionary containing fixed attributes of the flight (e.g., aircraft type and callsign). You do not need to reference this in your output.

2. `flight_data`: A list of time-ordered datapoints. Each datapoint is a list of 15 values in the following order:

[
  lat,                         # Latitude in decimal degrees
  lon,                         # Longitude in decimal degrees
  track,                       # Aircraft heading in degrees (0–360)
  alt,                         # Altitude in feet
  gspeed,                      # Ground speed in knots
  vspeed,                      # Vertical speed in feet per minute
  timestamp,                   # ISO 8601 UTC timestamp (e.g., "2025-04-01T04:31:59Z")
  edr_300hPa,                  # Eddy Dissipation Rate at 300 hPa (m²/s³), indicates clear-air turbulence
  ellrod_300hPa,               # Ellrod turbulence index at 300 hPa (s⁻²), reflects shear and deformation
  richardson_number_300hPa,    # Richardson number at 300 hPa (dimensionless); >10 = stable, <1 = unstable
  cape,                        # Convective Available Potential Energy (J/kg); indicates potential for storms/turbulence
  wind_speed_10000m,           # Wind speed at 10,000 meters in m/s
  wind_dir_10000m,             # Wind direction at 10,000 meters in degrees
  wind_speed_2m,               # Surface wind speed in m/s
  wind_dir_2m                  # Surface wind direction in degrees
]

Your task is to write a detailed and vivid narrative that describes the **entire flight** from beginning to end, as if told by a meteorological and flight operations expert. You must:

- **Refer to all time points relative to takeoff** as "X minutes after departure" instead of using timestamps or UTC times.
- **Do not mention the flight number, date, origin, or destination** — assume all flights follow the same Paris-to-Rome route.
- **Focus on the aircraft’s behavior**: climb rate, altitude, speed, changes in direction or vertical motion.
- **Incorporate meteorological observations**: turbulence (EDR, Ellrod), atmospheric stability (Richardson number), wind shear, and convective potential (CAPE).
- **Identify and name key geographic regions the flight passes over**, such as:
  - Île-de-France  
  - Burgundy (Bourgogne)  
  - the Massif Central  
  - the Alps  
  - the Ligurian coast  
  - western Tuscany  
  - Lazio and the Tyrrhenian Sea near Rome  
  You are encouraged to **infer likely geographic locations** based on the coordinates.

Write in a descriptive, coherent, and human-readable way — as if you're describing the flight path and conditions to a curious expert.
The description of the flight should be professional, not a touristic review narrative.


Here is the flight record:

{flight_data}

You should return the flight description along with the following flight_id: {flight_id} as a valid python list with the following structure:

[the flight id, your answer]

Your answer must not contain any unescaped newlines. 
"""

# Define the feature generation prompt template
flight_summary_prompt = ChatPromptTemplate.from_template(prompt)

# Define the pipeline (chain)
flight_summary_chain = (
        {
        "flight_data": lambda x: x,
        "flight_id": lambda x: x["flight_metadata"]["fr24_id"]
        }
    | flight_summary_prompt  # PromptTemplate formats the input variables
    | chat_model                    # Pass the prompt to the GPT-4o model
    | StrOutputParser()        # Parse the output into a clean string
  )

In [ ]:
import json
import re

def clean_answer(python_string):
    python_string = re.sub(r"^```(?:python|json)?\n", "", python_string.strip())
    python_string = python_string.strip().rstrip("```")
    return json.loads(python_string)

In [ ]:
BATCH_SIZE = 20
total = len(flights_data)
all_results = []
failed_flights = []

for batch_num in range(0, total, BATCH_SIZE):
    batch = flights_data[batch_num:batch_num + BATCH_SIZE]
    percent_complete = (batch_num + BATCH_SIZE) / total * 100
    print(f"Processing batch {batch_num // BATCH_SIZE + 1}: {percent_complete:.1f}% complete")
    result_batch = flight_summary_chain.batch(batch, {"max_concurrency": BATCH_SIZE})
    for r in result_batch:
        try:
            clean_result = clean_answer(r)
            all_results.append({"fr24_id": clean_result[0], "flight_summary": clean_result[1]})
        except:
            failed_flights.append(r[:20])
    


In [ ]:
len(all_results)

In [ ]:
# import json
# with open("flight_summary.json", "w") as f:
#     json.dump(all_results, f, indent=2)

NOW WE NEED TO RUN AGAIN THE FAILED FLIGHTS BY LOOKING AT THE FLIGHT_SUMMARY.JSON FILE, GETTING A LIST OF ALL THE FLIGHT_ID, AND PASSING THE FLIGHTS WITH FLIGHT_ID THAT ARE NOT IN THIS LIST

In [ ]:
import json

with open("final_data.json", "r") as f:
    flights_data = json.load(f)

with open("flight_summary.json", "r") as f:
    flights_summary = json.load(f)

existing_flight_id = [elem["fr24_id"] for elem in flights_summary]

flights_data_sample = [elem for elem in flights_data if elem["flight_metadata"]["fr24_id"] not in existing_flight_id]

BATCH_SIZE = 20
total = len(flights_data_sample)
all_results = []
failed_flights = []

for batch_num in range(0, total, BATCH_SIZE):
    batch = flights_data_sample[batch_num:batch_num + BATCH_SIZE]
    percent_complete = (batch_num + BATCH_SIZE) / total * 100
    print(f"Processing batch {batch_num // BATCH_SIZE + 1}: {percent_complete:.1f}% complete")
    result_batch = flight_summary_chain.batch(batch, {"max_concurrency": BATCH_SIZE})
    for r in result_batch:
        try:
            clean_result = clean_answer(r)
            all_results.append({"fr24_id": clean_result[0], "flight_summary": clean_result[1]})
        except:
            failed_flights.append(r[:20])

In [ ]:
flights_summary += all_results

import json
with open("flight_summary.json", "w") as f:
    json.dump(flights_summary, f, indent=2)